In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("satyamrajput7913/ais-ship-tracking-vessel-dynamics-and-eta-data")

print("Path to dataset files:", path)

100%|██████████| 68.8M/68.8M [00:04<00:00, 15.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/satyamrajput7913/ais-ship-tracking-vessel-dynamics-and-eta-data/versions/1


In [ ]:
%%writefile "/content/drive/MyDrive/ResearchProject/ais/preprocess_ais.py"

# Paste your entire preprocess_ais.py code below this line
import os
import pandas as pd

# ==========================================================
# AIS DATASET PATH (KaggleHub)
# ==========================================================

DATASET_PATH = "/root/.cache/kagglehub/datasets/satyamrajput7913/ais-ship-tracking-vessel-dynamics-and-eta-data/versions/1"

INPUT_FILE = os.path.join(DATASET_PATH, "processed_AIS_dataset.csv")

# ==========================================================
# OUTPUT PATH (Google Drive)
# ==========================================================

OUTPUT_DIR = "/content/drive/MyDrive/ResearchProject/ais"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "clean_ais.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)


class AISPreprocessor:

    def __init__(self, input_file):
        self.input_file = input_file

    def load_dataset(self):

        print("Loading AIS Dataset...\n")

        df = pd.read_csv(self.input_file)

        print(f"Total Records : {len(df)}")
        print(f"Total Columns : {len(df.columns)}")

        return df

    def preprocess(self, df):

        print("\nCleaning Dataset...")

        # Keep only useful columns
        df = df[
            [
                "MMSI",
                "BaseDateTime",
                "LAT",
                "LON",
                "SOG",
                "COG",
                "Heading",
                "VesselName",
                "VesselType",
                "Status"
            ]
        ]

        # Remove duplicate rows
        df = df.drop_duplicates()

        # Remove missing values
        df = df.dropna()

        # Convert timestamp
        df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])

        # Sort vessel history
        df = df.sort_values(
            by=["MMSI", "BaseDateTime"]
        )

        # Reset index
        df = df.reset_index(drop=True)

        print("Cleaning Completed")

        return df

    def save_dataset(self, df):

        df.to_csv(OUTPUT_FILE, index=False)

        print("\nDataset Saved Successfully")
        print(f"Location : {OUTPUT_FILE}")


# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":

    processor = AISPreprocessor(INPUT_FILE)

    df = processor.load_dataset()

    clean_df = processor.preprocess(df)

    processor.save_dataset(clean_df)

Writing /content/drive/MyDrive/ResearchProject/ais/preprocess_ais.py


In [ ]:
!python "/content/drive/MyDrive/ResearchProject/ais/preprocess_ais.py"

Loading AIS Dataset...

Total Records : 1098966
Total Columns : 28

Cleaning Dataset...
Cleaning Completed

Dataset Saved Successfully
Location : /content/drive/MyDrive/ResearchProject/ais/clean_ais.csv


In [ ]:
%%writefile /content/drive/MyDrive/ResearchProject/ais/parser.py

import pandas as pd
import os


class AISParser:
    """
    Loads and provides access to the cleaned AIS dataset.
    """

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.df = None

    def load_dataset(self):
        self.df = pd.read_csv(self.csv_path)

        self.df["BaseDateTime"] = pd.to_datetime(
            self.df["BaseDateTime"]
        )

        print("\nDataset Loaded Successfully")
        print(f"Total Records : {len(self.df)}")

        return self.df

    def get_all_vessels(self):
        return self.df

    def get_vessel_by_mmsi(self, mmsi):
        vessel = self.df[self.df["MMSI"] == mmsi]
        return vessel

    def get_latest_position(self, mmsi):
        vessel = self.get_vessel_by_mmsi(mmsi)

        if vessel.empty:
            return None

        latest = vessel.sort_values(
            by="BaseDateTime"
        ).iloc[-1]

        return {
            "MMSI": latest["MMSI"],
            "Latitude": latest["LAT"],
            "Longitude": latest["LON"],
            "Speed": latest["SOG"],
            "Course": latest["COG"],
            "Heading": latest["Heading"],
            "Status": latest["Status"],
            "Timestamp": latest["BaseDateTime"]
        }

    def get_unique_vessels(self):
        return self.df["MMSI"].nunique()

    def extract_los_angeles_region(self, output_file):
        """
        Extract vessels within the Los Angeles Port region.
        """

        la = self.df[
            (self.df["LAT"] >= 33.65) &
            (self.df["LAT"] <= 33.80) &
            (self.df["LON"] >= -118.30) &
            (self.df["LON"] <= -118.15)
        ].copy()

        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        la.to_csv(output_file, index=False)

        print("\nLos Angeles AIS dataset created")
        print(f"Records : {len(la)}")
        print(f"Saved to : {output_file}")

        return la


if __name__ == "__main__":
    INPUT = "/content/drive/MyDrive/ResearchProject/ais/clean_ais.csv"
    OUTPUT = "/content/drive/MyDrive/ResearchProject/ais/locations/los_angeles.csv"

    parser = AISParser(INPUT)
    parser.load_dataset()
    parser.extract_los_angeles_region(OUTPUT)

Writing /content/drive/MyDrive/ResearchProject/ais/parser.py


In [ ]:
import sys

sys.path.append("/content/drive/MyDrive/ResearchProject/ais")

In [ ]:
from parser import AISParser

CSV_PATH = "/content/drive/MyDrive/ResearchProject/ais/clean_ais.csv"

parser = AISParser(CSV_PATH)

parser.load_dataset()

print("Unique Vessels :", parser.get_unique_vessels())

print(parser.get_latest_position(366967100))


Dataset Loaded Successfully
Total Records : 1096472
Unique Vessels : 12750
None


In [ ]:
!python "/content/drive/MyDrive/ResearchProject/ais/parser.py"


Dataset Loaded Successfully
Total Records : 1096472

Los Angeles AIS dataset created
Records : 13278
Saved to : /content/drive/MyDrive/ResearchProject/ais/locations/los_angeles.csv


In [15]:
%%writefile /content/drive/MyDrive/ResearchProject/ais/tracker.py

import os
import numpy as np
import pandas as pd


class VesselTracker:

    def __init__(
        self,
        max_age=10,
        min_hits=2,
        iou_threshold=0.15,
        max_center_distance=150.0
    ):
        """
        max_age:
            Maximum number of frames a track can disappear
            before being removed.

        min_hits:
            Minimum number of successful detections required
            before a track is considered confirmed.

        iou_threshold:
            Minimum IoU for bounding-box matching.

        max_center_distance:
            Maximum center-to-center distance in pixels when
            IoU is insufficient.
        """

        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.max_center_distance = max_center_distance

        self.next_track_id = 1
        self.tracks = {}


    # =========================================================
    # IoU
    # =========================================================

    @staticmethod
    def calculate_iou(box1, box2):

        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])

        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        intersection_width = max(
            0,
            x2 - x1
        )

        intersection_height = max(
            0,
            y2 - y1
        )

        intersection = (
            intersection_width
            * intersection_height
        )

        area1 = max(
            0,
            box1[2] - box1[0]
        ) * max(
            0,
            box1[3] - box1[1]
        )

        area2 = max(
            0,
            box2[2] - box2[0]
        ) * max(
            0,
            box2[3] - box2[1]
        )

        union = (
            area1
            + area2
            - intersection
        )

        if union <= 0:
            return 0.0

        return intersection / union


    # =========================================================
    # Center distance
    # =========================================================

    @staticmethod
    def center_distance(
        cx1,
        cy1,
        cx2,
        cy2
    ):

        return np.sqrt(
            (cx2 - cx1) ** 2
            +
            (cy2 - cy1) ** 2
        )


    # =========================================================
    # Create track
    # =========================================================

    def create_track(
        self,
        detection,
        frame
    ):

        track_id = self.next_track_id

        self.next_track_id += 1

        self.tracks[track_id] = {

            "track_id": track_id,

            "class": detection["class"],

            "source": detection.get(
                "source",
                None
            ),

            "x1": detection["x1"],
            "y1": detection["y1"],
            "x2": detection["x2"],
            "y2": detection["y2"],

            "cx": detection["cx"],
            "cy": detection["cy"],

            "last_frame": frame,

            "age": 0,

            "hits": 1,

            "confirmed": (
                self.min_hits <= 1
            ),

            "previous_cx": detection["cx"],
            "previous_cy": detection["cy"],

            "movement_pixels": 0.0,

            "heading_deg": 0.0
        }

        return track_id


    # =========================================================
    # Update track
    # =========================================================

    def update_track(
        self,
        track_id,
        detection,
        frame
    ):

        track = self.tracks[track_id]

        previous_cx = track["cx"]
        previous_cy = track["cy"]

        current_cx = detection["cx"]
        current_cy = detection["cy"]

        dx = (
            current_cx
            - previous_cx
        )

        dy = (
            current_cy
            - previous_cy
        )

        movement = np.sqrt(
            dx ** 2
            +
            dy ** 2
        )

        # -----------------------------------------------------
        # Image heading
        #
        # atan2 gives direction in image coordinates.
        # Convert to degrees and normalize to 0-360.
        # -----------------------------------------------------

        if movement > 0:

            heading = np.degrees(
                np.arctan2(
                    dy,
                    dx
                )
            )

            heading = (
                heading + 360
            ) % 360

        else:

            heading = track[
                "heading_deg"
            ]

        track.update({

            "class": detection["class"],

            "source": detection.get(
                "source",
                track["source"]
            ),

            "x1": detection["x1"],
            "y1": detection["y1"],
            "x2": detection["x2"],
            "y2": detection["y2"],

            "previous_cx": previous_cx,
            "previous_cy": previous_cy,

            "cx": current_cx,
            "cy": current_cy,

            "last_frame": frame,

            "age": 0,

            "hits": track["hits"] + 1,

            "confirmed": (
                track["hits"] + 1
                >= self.min_hits
            ),

            "movement_pixels": movement,

            "heading_deg": heading
        })


    # =========================================================
    # Find best detection for track
    # =========================================================

    def find_best_match(
        self,
        track,
        detections,
        used_detections
    ):

        best_index = None
        best_score = -1

        track_box = [
            track["x1"],
            track["y1"],
            track["x2"],
            track["y2"]
        ]

        for index, detection in detections.iterrows():

            if index in used_detections:
                continue

            # -------------------------------------------------
            # Class must match
            # -------------------------------------------------

            if (
                str(track["class"])
                != str(detection["class"])
            ):
                continue

            detection_box = [
                detection["x1"],
                detection["y1"],
                detection["x2"],
                detection["y2"]
            ]

            iou = self.calculate_iou(
                track_box,
                detection_box
            )

            distance = self.center_distance(
                track["cx"],
                track["cy"],
                detection["cx"],
                detection["cy"]
            )

            # -------------------------------------------------
            # Matching rule
            # -------------------------------------------------

            valid = (
                iou >= self.iou_threshold
                or
                distance
                <= self.max_center_distance
            )

            if not valid:
                continue

            # -------------------------------------------------
            # Combined score
            # -------------------------------------------------

            distance_score = max(
                0.0,
                1.0
                -
                (
                    distance
                    /
                    self.max_center_distance
                )
            )

            score = (
                0.7 * iou
                +
                0.3 * distance_score
            )

            if score > best_score:

                best_score = score
                best_index = index

        return best_index, best_score


    # =========================================================
    # Process frame
    # =========================================================

    def process_frame(
        self,
        detections,
        frame
    ):

        used_detections = set()

        # -----------------------------------------------------
        # Existing tracks
        # -----------------------------------------------------

        existing_tracks = list(
            self.tracks.keys()
        )

        # -----------------------------------------------------
        # First match existing tracks
        # -----------------------------------------------------

        for track_id in existing_tracks:

            if track_id not in self.tracks:
                continue

            track = self.tracks[
                track_id
            ]

            best_index, best_score = (
                self.find_best_match(
                    track,
                    detections,
                    used_detections
                )
            )

            if best_index is not None:

                detection = detections.loc[
                    best_index
                ]

                self.update_track(
                    track_id,
                    detection,
                    frame
                )

                used_detections.add(
                    best_index
                )

            else:

                track["age"] += 1

        # -----------------------------------------------------
        # Create new tracks for unmatched detections
        # -----------------------------------------------------

        for index, detection in detections.iterrows():

            if index in used_detections:
                continue

            self.create_track(
                detection,
                frame
            )

        # -----------------------------------------------------
        # Remove old tracks
        # -----------------------------------------------------

        remove_ids = []

        for track_id, track in self.tracks.items():

            if track["age"] > self.max_age:

                remove_ids.append(
                    track_id
                )

        for track_id in remove_ids:

            del self.tracks[
                track_id
            ]

        # -----------------------------------------------------
        # Return tracks updated in this frame
        # -----------------------------------------------------

        frame_results = []

        for track_id, track in self.tracks.items():

            if track["last_frame"] != frame:
                continue

            # -------------------------------------------------
            # Only confirmed tracks
            # -------------------------------------------------

            if not track["confirmed"]:
                continue

            frame_results.append({

                "frame": frame,

                "track_id": track_id,

                "class": track["class"],

                "source": track["source"],

                "x1": track["x1"],
                "y1": track["y1"],
                "x2": track["x2"],
                "y2": track["y2"],

                "cx": track["cx"],
                "cy": track["cy"],

                "movement_pixels":
                    track[
                        "movement_pixels"
                    ],

                "heading_deg":
                    track[
                        "heading_deg"
                    ],

                "track_hits":
                    track["hits"]
            })

        return frame_results


    # =========================================================
    # Main tracking function
    # =========================================================

    def track(
        self,
        detections_file,
        output_file
    ):

        print(
            "Loading detections..."
        )

        detections = pd.read_csv(
            detections_file
        )

        print(
            "Total detections:",
            len(detections)
        )

        required_columns = [
            "frame",
            "class",
            "x1",
            "y1",
            "x2",
            "y2",
            "cx",
            "cy"
        ]

        for column in required_columns:

            if column not in detections.columns:

                raise ValueError(
                    f"Missing required column: "
                    f"{column}"
                )

        detections = detections.sort_values(
            by="frame"
        ).copy()

        results = []

        frames = sorted(
            detections["frame"].unique()
        )

        print(
            "Frames:",
            len(frames)
        )

        # -----------------------------------------------------
        # Process each frame
        # -----------------------------------------------------

        for counter, frame in enumerate(
            frames,
            start=1
        ):

            frame_detections = detections[
                detections["frame"] == frame
            ].copy()

            frame_results = self.process_frame(
                frame_detections,
                frame
            )

            results.extend(
                frame_results
            )

            if (
                counter == 1
                or counter % 20 == 0
                or counter == len(frames)
            ):

                print(
                    f"Processed frame "
                    f"{counter}/{len(frames)}"
                )

        # -----------------------------------------------------
        # Create dataframe
        # -----------------------------------------------------

        tracked = pd.DataFrame(
            results
        )

        # -----------------------------------------------------
        # Save
        # -----------------------------------------------------

        os.makedirs(
            os.path.dirname(output_file),
            exist_ok=True
        )

        tracked.to_csv(
            output_file,
            index=False
        )

        # -----------------------------------------------------
        # Statistics
        # -----------------------------------------------------

        print("\n======================================")
        print("TRACKING COMPLETED")
        print("======================================")

        print(
            "Input detections:",
            len(detections)
        )

        print(
            "Tracked detections:",
            len(tracked)
        )

        if len(tracked) > 0:

            print(
                "Unique tracks:",
                tracked["track_id"].nunique()
            )

            print(
                "Unique frames:",
                tracked["frame"].nunique()
            )

            print(
                "Confirmed tracks:",
                tracked["track_id"].nunique()
            )

        print(
            "\nOutput saved to:",
            output_file
        )

        return tracked


# ============================================================
# CONFIGURATION
# ============================================================

INPUT = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "fused_detections.csv"
)

OUTPUT = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "tracked_fused_objects.csv"
)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    tracker = VesselTracker(

        # Allow a vessel to disappear temporarily
        max_age=10,

        # Require at least 2 detections
        min_hits=2,

        # Minimum bounding-box overlap
        iou_threshold=0.15,

        # Maximum movement between frames
        max_center_distance=150.0
    )

    result = tracker.track(
        INPUT,
        OUTPUT
    )

    print("\nFirst 10 records:")

    if len(result) > 0:
        print(
            result.head(10)
        )
    else:
        print(
            "No confirmed tracks found."
        )

Overwriting /content/drive/MyDrive/ResearchProject/ais/tracker.py


In [16]:
import sys

sys.path.append("/content/drive/MyDrive/ResearchProject/ais")

In [17]:
!python "/content/drive/MyDrive/ResearchProject/ais/tracker.py"


Loading detections...
Total detections: 1969
Frames: 203
Processed frame 1/203
Processed frame 20/203
Processed frame 40/203
Processed frame 60/203
Processed frame 80/203
Processed frame 100/203
Processed frame 120/203
Processed frame 140/203
Processed frame 160/203
Processed frame 180/203
Processed frame 200/203
Processed frame 203/203

TRACKING COMPLETED
Input detections: 1969
Tracked detections: 1529
Unique tracks: 282
Unique frames: 197
Confirmed tracks: 282

Output saved to: /content/drive/MyDrive/ResearchProject/ais/tracked_fused_objects.csv

First 10 records:
   frame  track_id class  ... movement_pixels  heading_deg  track_hits
0      1         2  ship  ...      145.137713   344.382652           2
1      4         1  ship  ...      138.273131   188.635265           2
2      4         4  ship  ...      114.082800    26.413823           2
3      6         1  ship  ...      120.165743   183.616448           3
4      6         3  ship  ...      107.426589   141.602017           2
5

In [18]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/ResearchProject/ais/tracked_fused_objects.csv"
)

print("Tracked detections:", len(df))
print("Unique tracks:", df["track_id"].nunique())
print("Unique frames:", df["frame"].nunique())

print("\nTrack length distribution:")
print(
    df.groupby("track_id")
      .size()
      .describe()
)

Tracked detections: 1529
Unique tracks: 282
Unique frames: 197

Track length distribution:
count    282.000000
mean       5.421986
std        9.916517
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       64.000000
dtype: float64


In [19]:
import pandas as pd

path = "/content/drive/MyDrive/ResearchProject/ais/tracked_fused_objects.csv"

df = pd.read_csv(path)

track_stats = (
    df.groupby("track_id")
      .agg(
          frames=("frame", "nunique"),
          first_frame=("frame", "min"),
          last_frame=("frame", "max"),
          max_movement=("movement_pixels", "max")
      )
      .sort_values("frames", ascending=False)
)

print(track_stats.head(30))

          frames  first_frame  last_frame  max_movement
track_id                                               
13            64           11         201    149.817472
12            64           14         200    142.806997
6             62            7         203    146.359731
14            60            9         205    147.969583
17            53           10         203    146.274139
30            52           33         205    148.781713
58            39           66         201    136.922582
61            35           73         200    147.903964
66            33           71         205    149.912334
100           30          118         203    137.933469
49            28           61         159    149.693429
107           26          125         205    145.996882
64            25           69         195    149.622118
78            23           87         171    145.530293
80            23           89         160    147.889201
1             22            4          53    147

In [20]:
print("\nTracks lasting >= 5 frames:",
      (track_stats["frames"] >= 5).sum())

print("Tracks lasting >= 10 frames:",
      (track_stats["frames"] >= 10).sum())

print("Tracks lasting >= 20 frames:",
      (track_stats["frames"] >= 20).sum())


Tracks lasting >= 5 frames: 80
Tracks lasting >= 10 frames: 32
Tracks lasting >= 20 frames: 18


In [21]:
import pandas as pd
import os

FUSED = "/content/drive/MyDrive/ResearchProject/ais/fused_detections.csv"
OUTPUT = "/content/drive/MyDrive/ResearchProject/ais/tile_bounds_los_angeles.csv"

df = pd.read_csv(FUSED)

frames = sorted(df["frame"].dropna().astype(int).unique())

# Synthetic Los Angeles test region
BASE_LAT = 33.65
BASE_LON = -118.30

# Size represented by each test frame/tile
TILE_SIZE = 0.002

rows = []

for i, frame in enumerate(frames):

    r = i // 20
    c = i % 20

    rows.append({
        "frame": frame,
        "lat_min": BASE_LAT + r * TILE_SIZE,
        "lat_max": BASE_LAT + (r + 1) * TILE_SIZE,
        "lon_min": BASE_LON + c * TILE_SIZE,
        "lon_max": BASE_LON + (c + 1) * TILE_SIZE
    })

tile_bounds = pd.DataFrame(rows)

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

tile_bounds.to_csv(OUTPUT, index=False)

print("Tile bounds created successfully")
print("Unique frames:", len(frames))
print("Tile-bound rows:", len(tile_bounds))
print(tile_bounds.head())
print(tile_bounds.tail())
print("\nSaved to:", OUTPUT)

Tile bounds created successfully
Unique frames: 203
Tile-bound rows: 203
   frame  lat_min  lat_max  lon_min  lon_max
0      0    33.65   33.652 -118.300 -118.298
1      1    33.65   33.652 -118.298 -118.296
2      2    33.65   33.652 -118.296 -118.294
3      3    33.65   33.652 -118.294 -118.292
4      4    33.65   33.652 -118.292 -118.290
     frame  lat_min  lat_max  lon_min  lon_max
198    201   33.668   33.670 -118.264 -118.262
199    202   33.668   33.670 -118.262 -118.260
200    203   33.670   33.672 -118.300 -118.298
201    204   33.670   33.672 -118.298 -118.296
202    205   33.670   33.672 -118.296 -118.294

Saved to: /content/drive/MyDrive/ResearchProject/ais/tile_bounds_los_angeles.csv


In [22]:
%%writefile /content/drive/MyDrive/ResearchProject/ais/georeference.py

import os
import pandas as pd


class GeoReferencer:

    def __init__(self, tile_bounds_file):

        self.bounds = pd.read_csv(tile_bounds_file)

        # Make sure frame is integer
        self.bounds["frame"] = self.bounds["frame"].astype(int)

        print("Tile bounds loaded successfully")
        print("Number of frame bounds:", len(self.bounds))


    def pixel_to_geo(self, cx, cy, frame, image_width, image_height):

        # Find geographic bounds for this frame
        tile = self.bounds[
            self.bounds["frame"] == int(frame)
        ]

        if tile.empty:
            return None, None

        tile = tile.iloc[0]

        lat_min = tile["lat_min"]
        lat_max = tile["lat_max"]

        lon_min = tile["lon_min"]
        lon_max = tile["lon_max"]

        # Pixel X -> longitude
        lon = lon_min + (
            cx / image_width
        ) * (lon_max - lon_min)

        # Pixel Y -> latitude
        # Image Y increases downward,
        # geographic latitude increases upward.
        lat = lat_max - (
            cy / image_height
        ) * (lat_max - lat_min)

        return lat, lon


    def process(
        self,
        tracked_file,
        output_file,
        image_width=640,
        image_height=640
    ):

        df = pd.read_csv(tracked_file)

        print("Tracked detections:", len(df))

        latitudes = []
        longitudes = []

        for _, row in df.iterrows():

            lat, lon = self.pixel_to_geo(
                row["cx"],
                row["cy"],
                row["frame"],
                image_width,
                image_height
            )

            latitudes.append(lat)
            longitudes.append(lon)

        df["latitude"] = latitudes
        df["longitude"] = longitudes

        os.makedirs(
            os.path.dirname(output_file),
            exist_ok=True
        )

        df.to_csv(
            output_file,
            index=False
        )

        print("\nGeoreferencing completed successfully")
        print("Output saved to:")
        print(output_file)

        print("\nSample:")
        print(
            df[
                [
                    "frame",
                    "track_id",
                    "class",
                    "latitude",
                    "longitude"
                ]
            ].head()
        )

        return df


# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

TILE_BOUNDS = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "tile_bounds_los_angeles.csv"
)

TRACKED_FILE = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "tracked_fused_objects.csv"
)

OUTPUT_FILE = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "georeferenced_detections.csv"
)


# -------------------------------------------------------
# Run
# -------------------------------------------------------

if __name__ == "__main__":

    geo = GeoReferencer(
        TILE_BOUNDS
    )

    result = geo.process(
        TRACKED_FILE,
        OUTPUT_FILE
    )

    print("\nFirst 10 rows:")
    print(result.head(10))

Overwriting /content/drive/MyDrive/ResearchProject/ais/georeference.py


In [23]:
!python /content/drive/MyDrive/ResearchProject/ais/georeference.py

Tile bounds loaded successfully
Number of frame bounds: 203
Tracked detections: 1529

Georeferencing completed successfully
Output saved to:
/content/drive/MyDrive/ResearchProject/ais/georeferenced_detections.csv

Sample:
   frame  track_id class   latitude   longitude
0      1         2  ship  33.651652 -118.295586
1      4         1  ship  33.650390 -118.290504
2      4         4  ship  33.649721 -118.289945
3      6         1  ship  33.650414 -118.286879
4      6         3  ship  33.650797 -118.286381

First 10 rows:
   frame  track_id class source  ...  heading_deg  track_hits   latitude   longitude
0      1         2  ship  radar  ...   344.382652           2  33.651652 -118.295586
1      4         1  ship  radar  ...   188.635265           2  33.650390 -118.290504
2      4         4  ship  radar  ...    26.413823           2  33.649721 -118.289945
3      6         1  ship  radar  ...   183.616448           3  33.650414 -118.286879
4      6         3  ship  radar  ...   141.602017

In [26]:
%%writefile /content/drive/MyDrive/ResearchProject/ais/matcher.py

import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


# ============================================================
# Haversine distance
# ============================================================

def haversine_matrix(
    det_lat,
    det_lon,
    ais_lat,
    ais_lon
):
    R = 6371.0

    det_lat = np.radians(np.asarray(det_lat))
    det_lon = np.radians(np.asarray(det_lon))

    ais_lat = np.radians(np.asarray(ais_lat))
    ais_lon = np.radians(np.asarray(ais_lon))

    lat1 = det_lat[:, None]
    lon1 = det_lon[:, None]

    lat2 = ais_lat[None, :]
    lon2 = ais_lon[None, :]

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0, 1)

    return (
        2
        * R
        * np.arcsin(np.sqrt(a))
    )


# ============================================================
# Multi-vessel AIS Matcher
# ============================================================

class MultiVesselAISMatcher:

    def __init__(
        self,
        max_distance_km=5.0,
        reliable_distance_km=1.0,
        moderate_distance_km=2.0
    ):

        # Maximum distance allowed for an association
        self.max_distance_km = max_distance_km

        # Distance categories
        self.reliable_distance_km = reliable_distance_km
        self.moderate_distance_km = moderate_distance_km


    # ========================================================
    # Distance category
    # ========================================================

    def distance_category(self, distance):

        if distance <= self.reliable_distance_km:

            return "reliable"

        elif distance <= self.moderate_distance_km:

            return "moderate"

        elif distance <= self.max_distance_km:

            return "weak"

        else:

            return "unmatched"


    # ========================================================
    # Match one frame
    # ========================================================

    def match_frame(
        self,
        detections,
        ais
    ):

        if detections.empty or ais.empty:

            return []

        distances = haversine_matrix(
            detections["latitude"].to_numpy(),
            detections["longitude"].to_numpy(),
            ais["LAT"].to_numpy(),
            ais["LON"].to_numpy()
        )

        # ----------------------------------------------------
        # Only allow matches within max distance
        # ----------------------------------------------------

        cost = distances.copy()

        cost[
            cost > self.max_distance_km
        ] = 1e6

        # ----------------------------------------------------
        # One-to-one assignment
        # ----------------------------------------------------

        detection_indices, ais_indices = (
            linear_sum_assignment(cost)
        )

        matches = []

        for det_idx, ais_idx in zip(
            detection_indices,
            ais_indices
        ):

            distance = distances[
                det_idx,
                ais_idx
            ]

            if distance > self.max_distance_km:

                continue

            matches.append(
                (
                    det_idx,
                    ais_idx,
                    distance
                )
            )

        return matches


    # ========================================================
    # Main matching
    # ========================================================

    def match(
        self,
        detections_file,
        ais_file,
        output_file
    ):

        print("Loading detections...")

        detections = pd.read_csv(
            detections_file
        )

        print(
            "Total detection records:",
            len(detections)
        )

        print("Loading AIS data...")

        ais = pd.read_csv(
            ais_file
        )

        print(
            "Total AIS records:",
            len(ais)
        )

        # ----------------------------------------------------
        # Required columns
        # ----------------------------------------------------

        required_detection = [
            "frame",
            "latitude",
            "longitude"
        ]

        required_ais = [
            "MMSI",
            "LAT",
            "LON"
        ]

        for column in required_detection:

            if column not in detections.columns:

                raise ValueError(
                    f"Missing detection column: {column}"
                )

        for column in required_ais:

            if column not in ais.columns:

                raise ValueError(
                    f"Missing AIS column: {column}"
                )

        # ----------------------------------------------------
        # Numeric conversion
        # ----------------------------------------------------

        detections["latitude"] = pd.to_numeric(
            detections["latitude"],
            errors="coerce"
        )

        detections["longitude"] = pd.to_numeric(
            detections["longitude"],
            errors="coerce"
        )

        ais["LAT"] = pd.to_numeric(
            ais["LAT"],
            errors="coerce"
        )

        ais["LON"] = pd.to_numeric(
            ais["LON"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Remove invalid coordinates
        # ----------------------------------------------------

        detections = detections.dropna(
            subset=[
                "latitude",
                "longitude"
            ]
        ).copy()

        ais = ais.dropna(
            subset=[
                "LAT",
                "LON"
            ]
        ).copy()

        # ----------------------------------------------------
        # Remove duplicate AIS positions
        # ----------------------------------------------------

        ais = ais.drop_duplicates(
            subset=[
                "MMSI",
                "LAT",
                "LON"
            ]
        ).copy()

        print(
            "Valid AIS records:",
            len(ais)
        )

        print(
            "Unique AIS vessels:",
            ais["MMSI"].nunique()
        )

        # ----------------------------------------------------
        # Result storage
        # ----------------------------------------------------

        results = []

        frames = sorted(
            detections["frame"].unique()
        )

        print(
            "\nProcessing",
            len(frames),
            "frames..."
        )

        # ====================================================
        # FRAME-BY-FRAME MATCHING
        # ====================================================

        for counter, frame in enumerate(
            frames,
            start=1
        ):

            frame_detections = detections[
                detections["frame"] == frame
            ].copy()

            if frame_detections.empty:

                continue

            # ------------------------------------------------
            # Geographic candidate region
            # ------------------------------------------------

            lat_min = (
                frame_detections["latitude"].min()
                - 0.10
            )

            lat_max = (
                frame_detections["latitude"].max()
                + 0.10
            )

            lon_min = (
                frame_detections["longitude"].min()
                - 0.10
            )

            lon_max = (
                frame_detections["longitude"].max()
                + 0.10
            )

            frame_ais = ais[
                (ais["LAT"] >= lat_min)
                &
                (ais["LAT"] <= lat_max)
                &
                (ais["LON"] >= lon_min)
                &
                (ais["LON"] <= lon_max)
            ].copy()

            # ------------------------------------------------
            # No AIS candidates
            # ------------------------------------------------

            if frame_ais.empty:

                for _, detection in (
                    frame_detections.iterrows()
                ):

                    result = detection.to_dict()

                    result["MMSI"] = None
                    result["VesselName"] = None
                    result["VesselType"] = None

                    result["AIS_LAT"] = np.nan
                    result["AIS_LON"] = np.nan

                    result["SOG"] = np.nan
                    result["COG"] = np.nan

                    result["distance_to_ais_km"] = np.nan

                    result["match_quality"] = "unmatched"

                    result["ais_match"] = False

                    results.append(result)

                continue

            # ------------------------------------------------
            # One-to-one assignment
            # ------------------------------------------------

            matches = self.match_frame(
                frame_detections,
                frame_ais
            )

            matched_detection_indices = set()

            # ------------------------------------------------
            # Save matches
            # ------------------------------------------------

            for (
                det_idx,
                ais_idx,
                distance
            ) in matches:

                detection = (
                    frame_detections.iloc[
                        det_idx
                    ]
                )

                vessel = (
                    frame_ais.iloc[
                        ais_idx
                    ]
                )

                category = (
                    self.distance_category(
                        distance
                    )
                )

                result = detection.to_dict()

                result["MMSI"] = vessel.get(
                    "MMSI",
                    None
                )

                result["VesselName"] = vessel.get(
                    "VesselName",
                    None
                )

                result["VesselType"] = vessel.get(
                    "VesselType",
                    None
                )

                result["AIS_LAT"] = vessel["LAT"]
                result["AIS_LON"] = vessel["LON"]

                result["SOG"] = vessel.get(
                    "SOG",
                    None
                )

                result["COG"] = vessel.get(
                    "COG",
                    None
                )

                result["distance_to_ais_km"] = distance

                result["match_quality"] = category

                result["ais_match"] = True

                results.append(result)

                matched_detection_indices.add(
                    det_idx
                )

            # ------------------------------------------------
            # Save unmatched detections
            # ------------------------------------------------

            for det_idx in range(
                len(frame_detections)
            ):

                if det_idx in matched_detection_indices:

                    continue

                detection = (
                    frame_detections.iloc[
                        det_idx
                    ]
                )

                result = detection.to_dict()

                result["MMSI"] = None
                result["VesselName"] = None
                result["VesselType"] = None

                result["AIS_LAT"] = np.nan
                result["AIS_LON"] = np.nan

                result["SOG"] = np.nan
                result["COG"] = np.nan

                result["distance_to_ais_km"] = np.nan

                result["match_quality"] = "unmatched"

                result["ais_match"] = False

                results.append(result)

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            if (
                counter == 1
                or counter % 20 == 0
                or counter == len(frames)
            ):

                print(
                    f"Processed frame "
                    f"{counter}/{len(frames)}"
                )

        # ====================================================
        # OUTPUT
        # ====================================================

        matched = pd.DataFrame(
            results
        )

        sort_columns = [
            c
            for c in [
                "frame",
                "track_id"
            ]
            if c in matched.columns
        ]

        if sort_columns:

            matched = matched.sort_values(
                sort_columns
            )

        os.makedirs(
            os.path.dirname(output_file),
            exist_ok=True
        )

        matched.to_csv(
            output_file,
            index=False
        )

        # ====================================================
        # STATISTICS
        # ====================================================

        total = len(matched)

        matched_count = int(
            matched["ais_match"].sum()
        )

        unmatched_count = (
            total - matched_count
        )

        print("\n========================================")
        print("MULTI-VESSEL AIS MATCHING")
        print("========================================")

        print(
            "Total records:",
            total
        )

        print(
            "Matched:",
            matched_count
        )

        print(
            "Unmatched:",
            unmatched_count
        )

        if total > 0:

            print(
                "Match rate:",
                f"{matched_count / total * 100:.2f}%"
            )

        # ----------------------------------------------------
        # Quality statistics
        # ----------------------------------------------------

        print("\nMatch quality:")

        print(
            matched["match_quality"]
            .value_counts()
        )

        print("\nDistance statistics (km):")

        if matched_count > 0:

            print(
                matched.loc[
                    matched["ais_match"],
                    "distance_to_ais_km"
                ].describe()
            )

        # ----------------------------------------------------
        # MMSI statistics
        # ----------------------------------------------------

        print("\nUnique MMSI:")

        print(
            matched.loc[
                matched["ais_match"],
                "MMSI"
            ].nunique()
        )

        print("\nMMSI distribution:")

        print(
            matched.loc[
                matched["ais_match"],
                "MMSI"
            ].value_counts()
        )

        print(
            "\nOutput saved to:",
            output_file
        )

        return matched


# ============================================================
# CONFIGURATION
# ============================================================

GEOREFERENCED_FILE = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "georeferenced_detections.csv"
)

AIS_FILE = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "clean_ais.csv"
)

OUTPUT_FILE = (
    "/content/drive/MyDrive/ResearchProject/ais/"
    "matched_vessels.csv"
)


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    matcher = MultiVesselAISMatcher(

        # Hard maximum:
        # anything farther than 5 km is NOT matched
        max_distance_km=5.0,

        # <= 1 km = reliable
        reliable_distance_km=1.0,

        # 1-2 km = moderate
        moderate_distance_km=2.0
    )

    result = matcher.match(
        GEOREFERENCED_FILE,
        AIS_FILE,
        OUTPUT_FILE
    )

    print("\nFirst 10 records:")

    columns = [
        "frame",
        "track_id",
        "latitude",
        "longitude",
        "MMSI",
        "AIS_LAT",
        "AIS_LON",
        "distance_to_ais_km",
        "match_quality",
        "ais_match"
    ]

    columns = [
        c
        for c in columns
        if c in result.columns
    ]

    print(
        result[columns].head(10)
    )

Overwriting /content/drive/MyDrive/ResearchProject/ais/matcher.py


In [27]:
!pip install -q scipy

In [28]:
%cd /content/drive/MyDrive/ResearchProject/ais
!python matcher.py

/content/drive/.shortcut-targets-by-id/1dDE-zeDM0IWdqC_HJVf8Coat-kHTp87m/ResearchProject/ais
Loading detections...
Total detection records: 1529
Loading AIS data...
Total AIS records: 1096472
Valid AIS records: 615878
Unique AIS vessels: 12750

Processing 197 frames...
Processed frame 1/197
Processed frame 20/197
Processed frame 40/197
Processed frame 60/197
Processed frame 80/197
Processed frame 100/197
Processed frame 120/197
Processed frame 140/197
Processed frame 160/197
Processed frame 180/197
Processed frame 197/197

MULTI-VESSEL AIS MATCHING
Total records: 1529
Matched: 1520
Unmatched: 9
Match rate: 99.41%

Match quality:
match_quality
weak         631
moderate     466
reliable     423
unmatched      9
Name: count, dtype: int64

Distance statistics (km):
count    1520.000000
mean        1.933908
std         1.253673
min         0.013648
25%         0.908550
50%         1.681229
75%         2.778235
max         4.997656
Name: distance_to_ais_km, dtype: float64

Unique MMSI:
5

MM

In [25]:
import pandas as pd

path = "/content/drive/MyDrive/ResearchProject/ais/matched_vessels.csv"

df = pd.read_csv(path)

print("Total records:", len(df))
print("Matched:", df["ais_match"].sum())
print("Unmatched:", (~df["ais_match"]).sum())

print("\nMatch quality:")
print(df["match_quality"].value_counts())

print("\nUnique MMSI:")
print(df.loc[df["ais_match"], "MMSI"].nunique())

print("\nMMSI distribution:")
print(
    df.loc[df["ais_match"], "MMSI"].value_counts()
)

print("\nDistance statistics:")
print(
    df.loc[df["ais_match"], "distance_to_ais_km"].describe()
)

Total records: 1529
Matched: 1520
Unmatched: 9

Unique MMSI:
5

MMSI distribution:
MMSI
215347000.0    1188
367006770.0     134
366943940.0     111
368164540.0      86
366892000.0       1
Name: count, dtype: int64

Distance statistics:
count    1520.000000
mean        1.933908
std         1.253673
min         0.013648
25%         0.908550
50%         1.681229
75%         2.778235
max         4.997656
Name: distance_to_ais_km, dtype: float64
